# Tutorial: Managing Data in the FNQS Project

This notebook gives an example of how to manage data in a Neural Network Quantum States project, particularly for a Foundational model (FNQS).  
It covers how to save configurations, network weights, and MCMC samples, and how to reload them for analysis.  
We will simulate the training of a 1D Ising model.  
To understand the details of the model, please refer to [Other Notebook].

## Part 1: Saving the Data (During Training)
The way we save data consists of two separate formats: a "meta" JSON document, which contains the configuration of our Neural Network model and is easily readable for a human; and a .nk file containing the result of the training (the weights in the NN). This file is not meant to be read by a human, but it contains the actual information.

### Step 1.1: Saving the Configuration (`meta.json`)
We store the physics parameters and ViT architecture in a dictionary and save it to a JSON file.  
First, we decide the name and the path to the folder that will contain the data of our training run.  
`exist_ok=True` avoids any crash in case the folder already exists.

In [ ]:
import os
import json
import numpy as np

run_dir = "./Run_1D_L5_FNQS_Example" # Name and path of the directory containing the data
os.makedirs(run_dir, exist_ok=True)

Then we store the data of interest as a dictionary:

In [ ]:
meta = {
    "L": 8,
    "graph": "Chain 1D",
    "n_dim": 1,
    "hamiltonian": {
        "type": "Ising Disorder",
        "J": 1.0,
        "h0_train_list": [0.2, 0.6, 1.0, 1.5, 2.0],
        "sigma": 0.1
    },
    "model": "ViTFNQS",
    "vit_config": {
        "num_layers": 2,
        "d_model": 16,
        "heads": 4,
        "b": 2,
        "L_eff": 4
    },
    "nb_training_steps": 300,
    "total_configs_train": 10
}

Finally, we create the JSON file and "dump" our data inside (indent allows us to add line breaks so the file is easier to read):

In [ ]:
with open(os.path.join(run_dir, "meta.json"), "w") as f:
    json.dump(meta, f, indent=4)

In this example, the meta file contains mostly the structure of the model and the hyperparameters. However, the strength of this method is its ability to be personalized according to the needs of the project so that the data is always accompanied by any information that could be useful: to reproduce the training, to plot graphs, or to compare results with another training session. 

The following cell is a classic configuration of a ViT NN. 
Notice how we use the meta we defined as a dictionary to access data such as the number of spins L or the ViT hyperparameters.

In [ ]:
import netket as nk
import netket_foundation as nkf
import optax

hi = nk.hilbert.Spin(0.5, meta["L"])
ps = nkf.ParameterSpace(N=hi.size, min=0, max=10)

ma = nkf.model.ViTFNQS(**meta["vit_config"], n_coups=ps.size, complex=True, disorder=True)
sa = nk.sampler.MetropolisLocal(hi, n_chains=16)

total_configs = meta["total_configs_train"]
vs = nkf.FoundationalQuantumState(sa, ma, ps, n_replicas=total_configs, n_samples=1024)

rng = np.random.default_rng(1)
params_list = np.abs(rng.normal(loc=1.0, scale=meta["hamiltonian"]["sigma"], size=(total_configs, hi.size)))
vs.parameter_array = params_list

We noticed it was sometimes difficult to retrieve the exact values of the random replicas that were drawn, but this is still very useful information, for example to compare the accuracy of a training and the accuracy of a test of the same neural network.

To save the disorder configurations, we use NumPy rather than NetKet:

In [ ]:
np.save(os.path.join(run_dir, "disorder_configs.npy"), params_list)
def create_operator(params):
    ha_X = sum(params[i] * nkf.operator.sigmax(hi, i) for i in range(hi.size))
    ha_ZZ = sum(nkf.operator.sigmaz(hi, i) @ nkf.operator.sigmaz(hi, (i + 1) % hi.size) for i in range(hi.size))
    return -ha_X - meta["hamiltonian"]["J"] * ha_ZZ

ha_p = nkf.operator.ParametrizedOperator(hi, ps, create_operator)

optimizer = optax.sgd(0.01)
gs = nkf.VMC_NG(ha_p, optimizer, variational_state=vs)

We will now use the NetKet integrated logger. It consists of two types of files: 
- The JSON logger file that contains the values of observables of interest such as the energy, its variance, etc. `save_params = False` dictates that the weights in the network should not be stored in the JSON file (it would be too heavy); 
- The .nk files, which contain the optimized weights of our NN. Here, we decided to save every 10 steps, in case the training stops without reaching the total number of steps, so we can still work with the progress made.

In [ ]:
log = nk.logging.JsonLog(os.path.join(run_dir, "log_data"), save_params=False)
callback = nk.callbacks.SaveState(run_dir, prefix="state", save_every=10)

Now that we have defined where we want our logger to be, we can launch the training. The next cell runs the optimization process with the number of steps we defined in the meta, using the Hamiltonian we constructed in the first cells, and its output will be the logs and the .nk files.

In [ ]:
gs.run(n_iter=meta["nb_training_steps"], out=log, obs={"ham":ha_p}, callback=callback)

## Part 2. Using the data

### 2.1: Using already calculated data 
We can load the trained network and run a test through it to plot interesting observables.

In [ ]:
run_dir = "./Run_1D_L5_FNQS_Example" 
log = nk.logging.JsonLog(os.path.join(run_dir, "log_data.json"))

Here is a simple example to show how to plot the energy convergence, which is data that was already recorded.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

for i in range(meta["total_configs_train"]):
    energy = log.data["ham"][i].Mean.real
    iters = log.data["ham"][i].iters
    
    plt.plot(iters, energy, alpha=0.3, label=f"Rep {i}" if i < 5 else "")

plt.xlabel("Steps")
plt.ylabel("Energy $\\langle H \\rangle$")
plt.title("Energy convergence for all configurations")
plt.grid(True)
plt.show()

In [ ]:
import glob

checkpoints = glob.glob(os.path.join(run_dir, "*.nk"))
assert checkpoints, "No checkpoint file found."
checkpoint_path = sorted(checkpoints, key=lambda x: int(x.split("_")[-1].split(".")[0]))[-1]

We use the glob library here to search for every .nk file, then we sort them and take the state corresponding to the highest optimization step.
However, we could also have saved the number of optimization steps in the meta and loaded it here.

### 2.2: Using the trained network to compute a new observable
The first step consists of retrieving the structure of the network from the meta file.

In [ ]:
meta_path = os.path.join(run_dir, "meta.json")
with open(meta_path, "r") as f:
    meta = json.load(f)

L = meta["L"]
sigma_disorder = meta["hamiltonian"]["sigma"]
h0_train_list = meta["hamiltonian"]["h0_train_list"]
vit_params = meta["vit_config"]

Then, rebuild a new network with the same structure and hyperparameters. However, instead of initializing the weights randomly, we will use the data stored in the .nk files.  
That way, we obtain the Neural Network as it was optimized in the first part.

In [ ]:
hi = nk.hilbert.Spin(0.5, L)
ps = nkf.ParameterSpace(N=L, min=0, max=10)

ma_test = nkf.model.ViTFNQS(
    **meta["vit_config"],
    n_coups=ps.size,
    complex=True,
    disorder=True
)

sa_test = nk.sampler.MetropolisLocal(hi)
vs_test = nkf.FoundationalQuantumState(sa_test, ma_test, ps, n_replicas=meta["total_configs_train"])
vs_test.load(checkpoint_path)

replica_index = 0
pars_test = np.load(os.path.join(run_dir, "disorder_configs.npy"))[replica_index]

We will now plot the longitudinal magnetization:  
$M_z = \frac{1}{L} \sum_{i=1}^{L} \sigma_i^z$, a value of interest for the Ising Hamiltonian.  
For each $h_0$ value, we associate the mean of the replicas we drew and the mean of the magnetization values associated with each replica.

In [ ]:
mz_vals = []
h_vals = []

mz_op = sum(nkf.operator.sigmaz(hi, i) for i in range(meta["L"])) *(1.0 / meta["L"])

for i in range(meta["total_configs_train"]):
    pars = vs_test.parameter_array[i]
    _vs_test = vs_test.get_state(pars)
    
    res = _vs_test.expect(mz_op)
    mz_vals.append(res.Mean.real)
    h_vals.append(np.mean(pars))

plt.figure(figsize=(8, 5))
plt.scatter(h_vals, np.abs(mz_vals), color="red")
plt.xlabel("Average Transverse field $\\langle h \\rangle$")
plt.ylabel("Magnetization $|M_z|$")
plt.title("Phase transition: Magnetization vs Transverse field")
plt.grid(True, alpha=0.3)
plt.show()

### 2.3: Production Tips - Robust Loading and Data Persistence
In large-scale runs, weights are often saved in compressed archives. If `vs.load()` fails or if you need to manually inspect the weights, use the `zipfile` library.

In [ ]:
import zipfile
import flax

# Manual loading from a potentially zipped .nk file
if zipfile.is_zipfile(checkpoint_path):
    with zipfile.ZipFile(checkpoint_path, "r") as zf:
        # Find the largest .msgpack file (usually containing the variables)
        candidates = [f for f in zf.namelist() if f.endswith(".msgpack")]
        target = sorted(candidates, key=len)[-1]
        with zf.open(target) as f:
            state_dict = flax.serialization.msgpack_restore(f.read())
        
        # Restore variables into the state using from_state_dict for robustness
        vars_dict = state_dict.get("variables", state_dict.get("model", {}).get("variables", state_dict))
        vs_test.variables = flax.serialization.from_state_dict(vs_test.variables, vars_dict)
else:
    vs_test.load(checkpoint_path)

Another good practice for long analysis is to save results incrementally using `np.savez`. This ensures that if your notebook crashes, you don't lose the calculated observables.

In [ ]:
results_path = os.path.join(run_dir, "analysis_results.npz")
np.savez(results_path, 
         magnetization=np.array(mz_vals), 
         h_fields=np.array(h_vals), 
         meta=json.dumps(meta))
print(f"Results safely stored in {results_path}")